# Contrastive Loss

> Status: complete

## 题源线索

- Topic: InfoNCE / 对比学习损失。
- Source index: `source-research/niuke-ml-dl-topic-index.md`

## 手写实现约束

允许使用 Python 基础语法和 NumPy；不允许调用 sklearn、torch 或现成算法实现。

## 原理最小说明

InfoNCE 把正样本放在候选集合第 0 位，对候选相似度做 cross entropy：

$$L=-\log \frac{e^{sim(q,k^+)/\tau}}{e^{sim(q,k^+)/\tau}+\sum_j e^{sim(q,k^-_j)/\tau}}$$

## 带提示练习区

先按 TODO 补全下面的函数。这个版本保留实现台阶。

In [ ]:
import numpy as np

def contrastive_loss(query, positive, negatives, temperature=1.0):
    """TODO guided implementation."""
    # TODO 1: prepare inputs and check shapes
    # TODO 2: implement the core formula
    # TODO 3: handle edge cases and return result
    raise NotImplementedError

## 无提示练习区

撤掉提示后，独立实现同一个功能。

In [ ]:
import numpy as np

def contrastive_loss(query, positive, negatives, temperature=1.0):
    """TODO blank implementation."""
    raise NotImplementedError

## 测试区

运行：

```bash
python tests.py
```

Notebook 中可以在实现无提示函数后直接运行测试区代码。

In [ ]:
def test_contrastive_loss():
    q = np.array([[1.0, 0.0]])
    p = np.array([[1.0, 0.0]])
    neg = np.array([[[0.0, 1.0], [-1.0, 0.0]]])
    good = contrastive_loss(q, p, neg, temperature=0.5)
    bad = contrastive_loss(q, np.array([[0.0, 1.0]]), neg, temperature=0.5)
    assert good < bad

test_contrastive_loss()
print("All tests passed.")

## STOP HERE

请先完成带提示练习区和无提示练习区，再查看参考答案。

## 参考答案与解析

In [ ]:
import numpy as np

def _normalize(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

def contrastive_loss(query, positive, negatives, temperature=1.0):
    q = _normalize(np.asarray(query, dtype=np.float64))
    p = _normalize(np.asarray(positive, dtype=np.float64))
    n = _normalize(np.asarray(negatives, dtype=np.float64))
    pos = np.sum(q * p, axis=-1, keepdims=True)
    neg = np.einsum('bd,bkd->bk', q, n)
    logits = np.concatenate([pos, neg], axis=1) / temperature
    logits = logits - np.max(logits, axis=1, keepdims=True)
    log_probs = logits - np.log(np.sum(np.exp(logits), axis=1, keepdims=True))
    return np.mean(-log_probs[:, 0])

### 解析

1. 先归一化，用 cosine similarity。
2. 正样本 logit 放第 0 列。
3. 对候选集合做 cross entropy。
4. temperature 越小，分布越尖锐。

## 工程要点 / 面试追问

见 `notes.md`。